In [6]:
# Modules
import duckdb as db
import pandas as pd
import numpy as np
import jenkspy
from sklearn.metrics import cohen_kappa_score


In [2]:
# Load data

mr = db.query("SELECT * FROM 'model_result.parquet'").df()


In [3]:
mr

,bovine_id,farm_id,vig_date,contact_order,last_contact,exposure_duration,herd_size,route_size,p_mean_stochastic,p_std,p_min,p_max,p_ci_lower,p_ci_upper,cv,p_deterministic
0,B347,F439,2022-08-17 13:53:09.0000000,1,2021-04-19 00:00:00.000,1617,217.469021,5,0.244424,0.078709,0.084779,0.477551,0.112291,0.401474,0.322019,0.249816
1,B348,F95,2024-01-02 10:20:25.0000000,1,2021-11-24 00:00:00.000,768,40.757216,3,0.088742,0.031576,0.028570,0.191376,0.038222,0.154597,0.355821,0.090246
2,B447,F1302,2023-07-19 12:18:48.0000000,1,2023-07-17 00:00:00.000,1,11043.000000,3,0.000306,0.000114,0.000095,0.000694,0.000127,0.000549,0.373510,0.000307
3,B651,F16,2022-10-18 16:54:52.0000000,1,2022-10-14 00:00:00.000,1,32594.000000,3,0.000341,0.000127,0.000106,0.000775,0.000142,0.000613,0.373503,0.000343
4,B695,F1079,2021-12-08 13:26:02.0000000,2,2016-10-19 00:00:00.000,72,1909.875000,3,0.017687,0.006546,0.005522,0.039766,0.007417,0.031572,0.370110,0.017791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,B558,F1588,2025-05-08 11:27:18.0000000,2,2025-04-28 00:00:00.000,6,374.000000,3,0.001167,0.000436,0.000362,0.002648,0.000486,0.002094,0.373342,0.001173
1448,B558,F940,2025-05-08 11:27:18.0000000,1,2025-05-04 00:00:00.000,48,15742.500000,3,0.015106,0.005599,0.004711,0.034013,0.006329,0.026988,0.370617,0.015191
1449,B500,F1596,2021-07-27 16:16:05.0000000,4,2012-01-01 00:00:00.000,2950,23.228475,5,0.258025,0.082279,0.090130,0.499528,0.119262,0.421468,0.318880,0.266777
1450,B36,F248,2024-09-09 15:59:13.0000000,1,2016-04-26 00:00:00.000,3223,207.669563,7,0.419255,0.116901,0.160597,0.722783,0.209727,0.637348,0.278831,0.433363


In [3]:

# The 'Anchor' (Max Risk per Positive Animal)

# Anchor cases: just only one farm in the route

mrc = mr.query("route_size ==1")

max_risks = mrc.groupby('bovine_id')['p_deterministic'].max()

#  Sensitivity Threshold (T_High)

T_high = max_risks.quantile(0.05)

# Medium/Low 

remaining_contacts = mr[mr['p_deterministic'] < T_high]['p_deterministic']
T_medium = remaining_contacts.median()

print(f"--- Risk Thresholds ---")
print(f"HIGH RISK (> {T_high:.4f}):")
print(f"   Defined as the risk level exceeded by at least one contact in 95% of positive cases.")
print(f"MEDIUM RISK ({T_medium:.4f} - {T_high:.4f}):")
print(f"   defined as the upper 50% of the remaining non-critical contacts.")
print(f"LOW RISK (< {T_medium:.4f}):")
print(f"   Defined as the bottom 50% of background contacts.")

# Validate
def classify_risk(p):
    if p >= T_high: return "High"
    elif p >= T_medium: return "Medium"
    else: return "Low"

mr['risk_class'] = mr['p_deterministic'].apply(classify_risk)

mrcl = mr.copy()

--- Risk Thresholds ---
HIGH RISK (> 0.0518):
   Defined as the risk level exceeded by at least one contact in 95% of positive cases.
MEDIUM RISK (0.0034 - 0.0518):
   defined as the upper 50% of the remaining non-critical contacts.
LOW RISK (< 0.0034):
   Defined as the bottom 50% of background contacts.


In [16]:
# Sensitivity Analysis of risk classification


def generate_pert(min_val, mode_val, max_val, size, seed=42):
    rng = np.random.default_rng(seed)
    range_val = max_val - min_val
    alpha = 1 + 4 * (mode_val - min_val) / range_val
    beta = 1 + 4 * (max_val - mode_val) / range_val
    return min_val + rng.beta(alpha, beta, size) * range_val


lambda_min = 1.0e-5 
lambda_mode = 2.7e-5 
lambda_max = 8.0e-5 


deterministic_lambda = (lambda_min + 4 * lambda_mode + lambda_max) / 6

def calculate_p(lambda_array, exposure, herd_size):
    return 1 - np.exp(-lambda_array * exposure * np.log(herd_size))

def classify_anchor(df, p_col):
    mrc = df[df['route_size'] == 1]
    max_risks = mrc.groupby('bovine_id')[p_col].max()
    T_high = max_risks.quantile(0.05)
    remaining_contacts = df[df[p_col] < T_high][p_col]
    T_medium = remaining_contacts.median()
    conditions = [df[p_col] >= T_high, df[p_col] >= T_medium]
    return np.select(conditions, ['High', 'Medium'], default='Low')

results_df = mr[['bovine_id', 'farm_id', 'route_size', 'exposure_duration', 'herd_size']].copy()


results_df['p_deterministic'] = calculate_p(deterministic_lambda, results_df['exposure_duration'], results_df['herd_size'])
results_df['class_deterministic'] = classify_anchor(results_df, 'p_deterministic')
baseline_class = results_df['class_deterministic']

n_iterations = 5000
retention_rates = []
kappas = []

for i in range(n_iterations):

    random_lambdas = generate_pert(lambda_min, lambda_mode, lambda_max, size=len(results_df), seed=42+i)
    
    temp_df = results_df.copy()
    temp_df['p_stoch'] = calculate_p(random_lambdas, temp_df['exposure_duration'], temp_df['herd_size'])
    class_stoch = classify_anchor(temp_df, 'p_stoch')
    
    retention_rates.append(np.mean(baseline_class == class_stoch) * 100)
    kappas.append(cohen_kappa_score(baseline_class, class_stoch))

mean_retention = np.mean(retention_rates)
min_retention = np.min(retention_rates)
max_retention = np.max(retention_rates)
mean_kappa = np.mean(kappas)
min_kappa = np.min(kappas)

pct_above_90 = np.mean(np.array(retention_rates) >= 90) * 100

summary_df = pd.DataFrame({
    'Metric': [
        'Total Monte Carlo Iterations',
        'Mean Retention Rate (%)',
        'Minimum Retention Rate (%)',
        'Maximum Retention Rate (%)',
        'Iterations with >90% Retention (%)',
        'Mean Cohen\'s Kappa',
        'Minimum Cohen\'s Kappa'
    ],
    'Value': [
        n_iterations,
        f"{mean_retention:.2f}",
        f"{min_retention:.2f}",
        f"{max_retention:.2f}",
        f"{pct_above_90:.2f}",
        f"{mean_kappa:.4f}",
        f"{min_kappa:.4f}"
    ]
})

display(summary_df)

,Metric,Value
0,Total Monte Carlo Iterations,5000
1,Mean Retention Rate (%),93.15
2,Minimum Retention Rate (%),84.09
3,Maximum Retention Rate (%),95.80
4,Iterations with >90% Retention (%),95.84
5,Mean Cohen's Kappa,0.8773
6,Minimum Cohen's Kappa,0.7319


In [4]:
# Saving mrcl object

db.query("SELECT * FROM mrcl").to_parquet("risk_class.parquet")

In [5]:
# Risk classification Summary

risk_summary = mrcl.groupby('risk_class').agg(
    n_contacts=('p_deterministic', 'count'),
    mean_p=('p_deterministic', 'mean'),
    median_p=('p_deterministic', 'median'),
    min_p=('p_deterministic', 'min'),
    max_p=('p_deterministic', 'max')
).reindex(['High', 'Medium', 'Low'])

risk_summary['pct_of_total (%)'] = (risk_summary['n_contacts'] / risk_summary['n_contacts'].sum() * 100).round(2)

display(risk_summary)

,n_contacts,mean_p,median_p,min_p,max_p,pct_of_total (%)
risk_class,,,,,,
High,756,0.205698,0.177147,0.051837,0.752434,52.07
Medium,349,0.022506,0.020672,0.003447,0.051291,24.04
Low,347,0.000592,0.000239,0.000000,0.003408,23.90
